## Table of Contents

- [Introduction](#Introduction)
  - [Remarks](#Remarks)
- [Self-organising Map Code Block](#Self-organising-Map-Code-Block)

## Introduction

This notebook will try to evaluate the code black that lives in `train()` method below. We will benchmark the code, evaluate its complexity, and provide alternative ways to speed up the calculations. We will also provide guidence around how we can put the training code into production. Where relevant, we will also provide code snippets.

### Remarks
- Project root includes bare-bones setup initiated using [`uv`](https://docs.astral.sh/uv/getting-started/installation/). If you have `uv` and `make` installed, you can simply run `make install` to build the virtual environment. If you do not have `uv`, you can install the dependencies using `pip`.
- Feel free to run the code blocks as they are.

## Self-organising Map Code Block

Below is the code snippet that provides a way to calculate the weights of a self-organising mapping (SOM) given number of maximum iterations, width and height of the 2D output layer. Here are some initial observations:

- As is this method has `O(n**4)` complexity which is rather steep. For large number of iterations, and 2D grid size, this method will choke irrespective of which dimension is growing.
- There is no restriction, or validation around the function parameters. For example, you cannot really have negative iterations, or if you provide  an output layer of size 2x2, you will get a divide-by-zero error since `log(1) = 0`.
- The way code is presented is only for scripting.
- Usage of non-alphanumeric variable names is not really desirable.

In [ ]:
import numpy as np

In [ ]:
def train(input_data, n_max_iterations, width, height):
    σ0 = max(width, height) / 2
    α0 = 0.1
    weights = np.random.random((width, height, 3))
    λ = n_max_iterations / np.log(σ0)
    for t in range(n_max_iterations):
        σt = σ0 * np.exp(-t/λ)
        αt = α0 * np.exp(-t/λ)
        for vt in input_data:
            bmu = np.argmin(np.sum((weights - vt) ** 2, axis=2))
            bmu_x, bmu_y = np.unravel_index(bmu, (width, height))
            for x in range(width):
                for y in range(height):
                    di = np.sqrt(((x - bmu_x) ** 2) + ((y - bmu_y) ** 2))
                    θt = np.exp(-(di ** 2) / (2*(σt ** 2)))
                    weights[x, y] += αt * θt * (vt - weights[x, y])
    return weights

In [7]:
input_data = np.random.random((10,3))
image_data = train(input_data, 10000, 10, 2)